## <font color='blue'>Projeto 7</font>
## <font color='blue'>Criando Memória Externa Para o LLM com Vector Database</font>

## Instalando e Carregando Pacotes

In [1]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 7.0 MB/s eta 0:00:00


In [2]:
!pip install -q unstructured_client unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 27.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 60.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 29.9 MB/s eta 0:00:00


In [3]:
!pip install -q langchain langchain-community transformers accelerate bitsandbytes sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.7/973.7 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 31.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.9/307.9 kB 30.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.4/121.4 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 51.4 MB/s eta 0:00:00


In [4]:
!pip install -q faiss-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 16.3 MB/s eta 0:00:00


In [5]:
# Imports
import unstructured_client
import unstructured
import langchain
import transformers
import sentence_transformers
import accelerate
import bitsandbytes
import faiss

In [6]:
# Imports
import os
import torch
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError
from unstructured.staging.base import dict_to_elements
from langchain_core.documents import Document
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from transformers import pipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from huggingface_hub.hf_api import HfFolder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

## Usando Ferramenta de Tratamento de Dados Não Estruturados

Veja o procedimento para criar a API no Capítulo 16 do curso.

In [8]:
# Define a chave
os.environ["UNSTRUCTURED_API_KEY"] = "coloque-aqui-sua-api"

In [9]:
# Cria a variável Python
unstructured_api_key = os.environ.get("UNSTRUCTURED_API_KEY")

In [10]:
# Cria o cliente
client = UnstructuredClient(api_key_auth = unstructured_api_key)

## Extraindo Dados de Texto de Arquivos PDF

In [11]:
# Caminho do arquivo PDF
caminho_do_arquivo = "arquivos/ArtigoDSA1.pdf"

In [12]:
# Abre o arquivo em modo de leitura binária
with open(caminho_do_arquivo, "rb") as f:

    # Cria um objeto com o conteúdo do arquivo PDF
    files = shared.Files(content = f.read(), file_name = caminho_do_arquivo)

    # Particiona o arquivo em pedaços (chunks)
    req = shared.PartitionParameters(files = files, chunking_strategy = "by_title", max_characters = 512)

    # Faz a requisição para a API e imprime o erro (se ocorrer)
    try:
        resp = client.general.partition(req)
    except SDKError as e:
        print(e)

In [13]:
# Obtém os elementos da resposta da API
elementos = dict_to_elements(resp.elements)

In [14]:
# Visualiza
elementos

## Carregando os Vetores dos Dados de Texto no Banco de Dados Vetorial

In [15]:
# Lista para os documentos
documentos = []

In [16]:
# Para cada elemento, extrai os metadados e os documentos
for elemento in elementos:
    metadados = elemento.metadata.to_dict()
    documentos.append(Document(page_content = elemento.text, metadata = metadados))

In [17]:
# Visualiza
metadados

{'filetype': 'application/pdf',
 'languages': ['eng'],
 'page_number': 3,
 'orig_elements': 'eJy1UsFu3CAQ/RXEeUGADbZ7S9QeU0Xa3LbRCsN4l8oG12a72Ub994J3o1RNL4mS43szj5l5j80jhh4G8HHrLP6EMBPCCM4KIlVbkVKqhtSWVcRURtQgyrbpFF4hPEDUVkedNI/YhDBZ53WEecG9PoVD3O7B7fYxMUIwljQX+uhs3CeWVws7Budj1m02qY3yFSobRdn9Cj1hWVwwF1JQ9h/irEgMnk9zhCFfcuseoF+P2gD+nQoWIpjogt+aXs/zdpxCm9oYrUteydTQuR7iaYRFe3uDl4X97qB3y1UbDH6H84gxMVt/GFqYEl/kxyM85Dvxt4Ng3Kzhu0bXYUAwoKtxAm9hWiqWohvtI/i9JjPk8hBsQCb4OWYeWUB6Ebhf2gaad3ha6aueJh3dT7jLs9LQf5NTrDN123XEgipIckSTRrctgUaWJZON5FZ/aHI5mLp+Ti5hJSQtMi5UQcVLfO5/W26VUpK/U25ffhzcCOjz+upVlndNaSyDgoDRLSlFVZKmVUBAcdF2TV3x2nyY5Tz/+WRpVZ0tveCKX7CqJVUv8bn/bZaXdV28k+PH45FmS2bjwBvQRlsYTtSEgbbT3yHcudinte7/APkUY4o=',
 'filename': 'ArtigoDSA1.pdf'}

In [18]:
# Visualiza
documentos

[Document(page_content='A Habilidade Mais Importante na Era da Inteligência Artificial\n\nA pandemia do COVID-19 acelerou o ritmo do desenvolvimento digital em todo o mundo, já que tudo, desde reuniões até consultas médicas, ficou online. Isso pode soar como algo super positivo.\n\nPara dezenas de milhões de trabalhadores, não.', metadata={'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'orig_elements': 'eJy1U01v1DAQ/SujnDeR87WJuVWAxB4KlUBclmo1iWezRo4dbKe0VPx3JlnaA6qQqOjN741nPOP3Zn+fkKGRbDxolbyCpCiaUtZdnbayx7SqC0y7rs1TKVTXlqKR2G6TDSQjRVQYkXPuk945r7TFSGHFBu/cHA8n0sMpMlMUQnDOb/q7VvHEbN6s7OS0jUvefp/LJis3UIgia6438IirByyKNmufItYMZpJwFyKNyyRX+pbMxwl7Sn5yQFGkPmpnD73BEA6Tdx1fE1lb1YJLJEdtKN5NtOZeXSZrw3aYcVin2idkh2R5YmLmYOexI79MsRSPdLvMmVzAO+y00QoVwSXqALtxcj6ijQQW4a1HUAg7hkYPX2YhCG2vES581EfNJ7O8+9DGe/Qeo76hT0t9fuhPtaQqt13bdynKfptWmLcpYi7TvKRSNU19zEX+gmpVWcN/3+ZZeVZrxZWoMrHiWjSZeII4ZzxPLVnmYvvf1JrQKhr5/5WD1x8+796kuQTuwZB3MzjwOo5uCSoKZG+cudHL7zPWg45ogEaIjuMOxtkqt4Gvq6g5fJsJ4

https://huggingface.co/BAAI/bge-m3

https://huggingface.co/BAAI/bge-base-en-v1.5

In [19]:
# Cria o VectorDB FAISS a partir dos documentos usando um modelo de embeddings do HF
vector_db_dsa = FAISS.from_documents(documentos,
                                     HuggingFaceEmbeddings(model_name = "BAAI/bge-m3"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [20]:
type(vector_db_dsa)

langchain_community.vectorstores.faiss.FAISS

In [21]:
# Cria o objeto retriever para recuperar dados do banco de dados vetorial
retriever = vector_db_dsa.as_retriever(search_type = "similarity",
                                       search_kwargs = {"k": 4})

## Carregando o LLM Open-Source com Processo de Quantização

Veja o procedimento para acessar o Llama 3 no Capítulo 16 do curso.

In [22]:
# Define a API
HfFolder.save_token('coloque-aqui-sua-chave-do-HF')

https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

In [23]:
# Nome do LLM
nome_llm = "meta-llama/Meta-Llama-3-8B-Instruct"

In [24]:
# Parâmetros de quantização
bnb_config = BitsAndBytesConfig(load_in_4bit = True,
                                bnb_4bit_use_double_quant = True,
                                bnb_4bit_quant_type = "nf4",
                                bnb_4bit_compute_dtype = torch.bfloat16)

In [25]:
# Carrega o LLM
modelo = AutoModelForCausalLM.from_pretrained(nome_llm, quantization_config = bnb_config)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [26]:
# Carrega o Tokenizador
tokenizador = AutoTokenizer.from_pretrained(nome_llm)

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## Preparando o Pipeline de Geração de Texto

In [27]:
# Cria os token terminators
tokens_terminators = [tokenizador.eos_token_id,
                      tokenizador.convert_tokens_to_ids("<|eot_id|>")]

In [28]:
# Cria o pipeline de geração de texto
text_generation_pipeline = pipeline(model = modelo,
                                    tokenizer = tokenizador,
                                    task = "text-generation",
                                    temperature = 0.2,
                                    do_sample = True,
                                    repetition_penalty = 1.1,
                                    return_full_text = False,
                                    max_new_tokens = 200,
                                    eos_token_id = tokens_terminators)

In [29]:
# Cria o Hugging Face Pipeline
llm = HuggingFacePipeline(pipeline = text_generation_pipeline)

## Definindo o Prompt Template com LangChain

Este será o formato do prompt usado como entrada para o LLM:

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{{ system_prompt }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ user_msg_1 }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ model_answer_1 }}<|eot_id|>
```

In [30]:
# Cria a variável com o texto para o PromptTemplate
dsa_prompt = """
<|start_header_id|>user<|end_header_id|>
Você é um assistente para tirar dúvidas sobre Inteligência Artificial.
Você recebe as partes extraídas de um documento longo e uma pergunta. Forneça uma resposta coloquial.
Se você não souber a resposta, basta dizer “Não sei”. Não invente uma resposta. Responda em português do Brasil.
Pergunta: {question}
Contexto: {context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

In [31]:
# Cria o objeto PromptTemplate
prompt = PromptTemplate(input_variables = ["context", "question"],
                        template = dsa_prompt)

In [32]:
# Cria a chain com Prompt e LLM
llm_chain = prompt | llm | StrOutputParser()

In [33]:
# Cria a chain final com o Retriever do Banco de Dados Vetorial e a chain do LLM
dsa_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain

Em sistemas de recuperação de informações, especialmente em aplicações que envolvem buscas em bancos de dados ou grandes conjuntos de dados, um "Retriever" é responsável por localizar e recuperar os dados relevantes com base em uma consulta fornecida. No contexto de bancos de dados vetoriais, esse retriever está configurado para converter dados de texto em vetores e realizar buscas semânticas nos dados indexados.

RunnablePassthrough por si só permite que você passe entradas inalteradas. Esta função serve como um pass-through, ou seja, ela simplesmente passa a entrada recebida para a saída sem fazer alterações. Isso pode ser útil para manter a consistência da interface de programação ou para realizar validações e formatações sem alterar os dados.

## Usando o LLM Para Responder Perguntas Usando o VectorDB como Fonte de Consulta

In [34]:
pergunta1 = "O que a pandemia do Covid-19 gerou no desenvolvimento digital?"

In [35]:
dsa_chain.invoke(pergunta1)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'Excelente pergunta!\n\nA pandemia do Covid-19 gerou um impulso significativo no desenvolvimento digital, pois muitas atividades que antes eram realizadas presencialmente foram transferidas para o ambiente online. Isso inclui reuniões, consultas médicas, compras e outras interações sociais.\n\nEssa mudança foi necessária para garantir a segurança pública e evitar a propagação do vírus, mas também trouxe desafios para muitas empresas e indivíduos. Por exemplo, muitas empresas tiveram que adaptar rapidamente suas operações para funcionar remotamente, o que demandou habilidades digitais específicas, como conhecimento em ferramentas de colaboração online e comunicação eficaz por meio de plataformas virtuais.\n\nAlém disso, a pandemia também expôs a falta de habilidades digitais em muitas áreas, especialmente entre os trabalhadores'

In [36]:
pergunta2 = "Quanto as principais economias do mundo podem perder até 2028 em crescimento potencial?"

In [37]:
dsa_chain.invoke(pergunta2)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'Não sei. A informação fornecida não menciona explicitamente o valor exato que as principais economias do mundo podem perder em crescimento potencial até 2028. Além disso, a fonte citada (Accenture) estima que a lacuna de habilidades pode levar a uma perda de US$ 11,5 trilhões em crescimento potencial até 2028, mas não especifica o valor que as principais economias do mundo podem perder individualmente. Se você tiver mais informações ou contexto adicional, posso tentar ajudar melhor.'

In [39]:
#%watermark -v -m

In [40]:
#%watermark --iversions

# Fim